In [ ]:
# -----------------------------
# 🛠️ 切换工作目录 & 添加模块路径
# -----------------------------
import os, sys
os.chdir(r"D:\Pycharm_project\covid_dashboard")  # 改成你的项目根目录
sys.path.append(os.getcwd())

# -----------------------------
# 📥 导入模块 & 加载数据
# -----------------------------
import pandas as pd
import plotly.express as px
from ipywidgets import interact, widgets
from src.data_load import load_data

df = load_data()

# -----------------------------
# 🔎 数据范围 & 排除汇总行
# -----------------------------
print("数据日期范围：", df['date'].min(), "到", df['date'].max())

exclude = ['World', 'Asia', 'Europe', 'European Union (27)',
           'High-income countries', 'Upper-middle-income countries',
           'Low-income countries', 'International',
           'Lower-middle-income countries', 'North America', 'South America']

df_countries = df[~df['location'].isin(exclude)]

# 每个国家取最后一条非零累计病例记录
latest_per_country = df_countries[df_countries['total_cases'] > 0].groupby('location').last().reset_index()

# 累计病例前10国家
top_countries = latest_per_country.sort_values('total_cases', ascending=False).head(10)['location'].tolist()

print("累计病例前10国家：")
print(latest_per_country[latest_per_country['location'].isin(top_countries)][['location','total_cases','new_cases']])

# -----------------------------
# 🌍 全球每日新增病例折线图
# -----------------------------
daily_total = df.groupby('date')['new_cases'].sum().reset_index()
fig_global = px.line(daily_total, x='date', y='new_cases', title='Global Daily New Cases')
fig_global.show()

# -----------------------------
# 🏆 前10国家累计病例折线图
# -----------------------------
df_top = df[df['location'].isin(top_countries)]
fig_top = px.line(df_top, x='date', y='total_cases', color='location',
                  title='Top 10 Countries - Total Cases Over Time')
fig_top.show()

# -----------------------------
# 🎛️ 交互式选择国家绘图
# -----------------------------
country_list = sorted(df_countries['location'].unique())

def plot_countries(selected_countries):
    df_plot = df_countries[df_countries['location'].isin(selected_countries)]
    fig = px.line(df_plot, x='date', y='total_cases', color='location',
                  title='Selected Countries - Total Cases Over Time')
    fig.show()

interact(plot_countries, selected_countries=widgets.SelectMultiple(
    options=country_list,
    value=['United States', 'India'],  # 默认选中的国家
    description='Countries',
    rows=10
))
